# Boundary-sensitivity resolution analysis

The **boundary sensitivities** live in the *same* NetCDF object as the surface footprint (`fp`).
They are stored as four variables, one per domain edge:

| variable | long name | dims |
|---|---|---|
| `particle_locations_n` | Fraction of total particles leaving domain (N side) | `(height, lon, time)` |
| `particle_locations_s` | Fraction of total particles leaving domain (S side) | `(height, lon, time)` |
| `particle_locations_e` | Fraction of total particles leaving domain (E side) | `(height, lat, time)` |
| `particle_locations_w` | Fraction of total particles leaving domain (W side) | `(height, lat, time)` |

Each value is the fraction of particles exiting the domain through a given edge, resolved
along that edge (longitude for N/S, latitude for E/W) and over `height`. They are convolved
with CAMS boundary-condition mole fractions (`vmr_n/s/e/w`) to give the background, see
[`gates/data/load_background_data.py`](../gates/data/load_background_data.py) `calculate_bg`.

This notebook reports the **resolution** of those sensitivities and of the CAMS boundary
conditions:
- Sections 1-4: footprint boundary sensitivities (along-edge + vertical), vs the footprint grid.
- Section 5: CAMS mole-fraction curtains (`vmr_*`) - levels, lat/lon and temporal resolution.
- Section 6: combined summary table.

> Principles touched: *honest data handling* (we read the resolution directly from the files,
> not from assumptions) and *reproducibility* (the file paths and env are stated explicitly).

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# A representative footprint file (same domain/object that holds the boundary sensitivities).
# Change this to any GOSAT-*_<DOMAIN>_<YYYYMM>.nc file under the fp data dir.
FP_FILE = "/group/chem/acrg/LPDM/fp_NAME_pre20210701/SOUTHAMERICA/GOSAT-BRAZIL-column_SOUTHAMERICA_201201.nc"

BOUNDARY_VARS = [
    "particle_locations_n",
    "particle_locations_s",
    "particle_locations_e",
    "particle_locations_w",
]

ds = xr.open_dataset(FP_FILE)
print("Dataset dimensions:", dict(ds.sizes))
print("Variables present:", [v for v in BOUNDARY_VARS if v in ds])

## 1. Grid resolution from the coordinate axes

Resolution = spacing between adjacent coordinate values. We report it for `lat`, `lon` and
`height`, and flag whether the spacing is uniform.

In [ ]:
def axis_resolution(ds, name):
    """Return a dict summarising the resolution of a 1-D coordinate axis."""
    vals = np.asarray(ds[name].values, dtype=float)
    steps = np.diff(vals)
    uniform = bool(np.allclose(steps, steps[0], atol=1e-6)) if len(steps) else True
    return {
        "n_points": len(vals),
        "min": vals.min(),
        "max": vals.max(),
        "step_median": float(np.median(steps)) if len(steps) else np.nan,
        "step_min": float(steps.min()) if len(steps) else np.nan,
        "step_max": float(steps.max()) if len(steps) else np.nan,
        "uniform": uniform,
        "units": ds[name].attrs.get("units", ""),
    }

for axis in ["lat", "lon", "height"]:
    if axis in ds:
        info = axis_resolution(ds, axis)
        print(f"{axis:>7}: {info['n_points']:>4} pts | "
              f"range [{info['min']:.3f}, {info['max']:.3f}] {info['units']} | "
              f"step ~{info['step_median']:.4f} "
              f"(min {info['step_min']:.4f}, max {info['step_max']:.4f}) | "
              f"uniform={info['uniform']}")

## 2. Resolution of each boundary-sensitivity variable

N/S edges run along longitude; E/W edges run along latitude. Every edge also carries a vertical
(`height`) dimension. So the "resolution" of a boundary sensitivity has two parts: the along-edge
horizontal spacing and the vertical spacing.

In [ ]:
edge_horizontal_axis = {
    "particle_locations_n": "lon",
    "particle_locations_s": "lon",
    "particle_locations_e": "lat",
    "particle_locations_w": "lat",
}

height_info = axis_resolution(ds, "height")
print(f"Vertical (height): {height_info['n_points']} levels, "
      f"{height_info['min']:.0f}-{height_info['max']:.0f} m, "
      f"step ~{height_info['step_median']:.0f} m, uniform={height_info['uniform']}\n")

for var, hax in edge_horizontal_axis.items():
    if var not in ds:
        continue
    da = ds[var]
    h = axis_resolution(ds, hax)
    print(f"{var}")
    print(f"    dims              : {da.dims}  shape {tuple(da.shape)}")
    print(f"    long_name         : {da.attrs.get('long_name', '')}")
    print(f"    along-edge axis   : {hax} | {h['n_points']} pts | "
          f"step ~{h['step_median']:.4f} deg | range [{h['min']:.2f}, {h['max']:.2f}]")
    print(f"    vertical axis     : height | {height_info['n_points']} levels | "
          f"step ~{height_info['step_median']:.0f} m\n")

## 3. Comparison with the footprint grid

The surface footprint `fp` is 2-D over `(lat, lon)` with **no** height dimension. The boundary
sensitivities re-use the footprint's horizontal spacing along their respective edge, and add a
vertical dimension the footprint does not have.

In [ ]:
fp = ds["fp"]
print(f"fp dims  : {fp.dims}  shape {tuple(fp.shape)}")
lat_info = axis_resolution(ds, "lat")
lon_info = axis_resolution(ds, "lon")
print(f"fp horizontal resolution: lat step ~{lat_info['step_median']:.4f} deg, "
      f"lon step ~{lon_info['step_median']:.4f} deg")
print("\nThe N/S boundary sensitivities share the lon spacing; E/W share the lat spacing.")
print("Difference: boundary sensitivities replace one horizontal axis with `height` "
      f"({height_info['n_points']} levels x {height_info['step_median']:.0f} m).")

## 4. Visualise a single time slice

Curtain plots (height vs along-edge position) for one footprint, so the along-edge and vertical
resolution are visible directly.

In [ ]:
t_idx = 0  # which footprint (time index) to plot
fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)

for ax, (var, hax) in zip(axes.ravel(), edge_horizontal_axis.items()):
    if var not in ds:
        ax.set_visible(False)
        continue
    da = ds[var].isel(time=t_idx)
    da.plot(ax=ax, x=hax, y="height", cbar_kwargs={"label": "particle fraction"})
    ax.set_title(f"{var}\n{ds[var].attrs.get('long_name', '')}")
    ax.set_xlabel(f"{hax} (deg)")
    ax.set_ylabel("height (m)")

fig.suptitle(f"Boundary sensitivities at time = {str(ds.time.values[t_idx])[:19]}", fontsize=13)
plt.show()

## 6. Summary table

Both the footprint boundary sensitivities and the CAMS mole-fraction curtains, side by side.

In [ ]:
# CAMS boundary-condition curtains (vmr_n/s/e/w), one file per month.
# Change to any ch4_<DOMAIN>_<YYYYMM>_CAMS-inversion.nc under the bc data dir.
CAMS_FILE = "/group/chem/acrg/LPDM/bc/SOUTHAMERICA/ch4_SOUTHAMERICA_201201_CAMS-inversion.nc"

cams = xr.open_dataset(CAMS_FILE)
print("CAMS dimensions:", dict(cams.sizes))
print("CAMS data vars :", list(cams.data_vars))
print()

for axis in ["lat", "lon", "height"]:
    if axis in cams:
        info = axis_resolution(cams, axis)
        print(f"{axis:>7}: {info['n_points']:>4} pts | "
              f"range [{info['min']:.3f}, {info['max']:.3f}] | "
              f"step ~{info['step_median']:.4f} | uniform={info['uniform']}")

cams_edge_axis = {"vmr_n": "lon", "vmr_s": "lon", "vmr_e": "lat", "vmr_w": "lat"}
print()
for var, hax in cams_edge_axis.items():
    if var in cams:
        print(f"{var}: dims {cams[var].dims}  shape {tuple(cams[var].shape)}  "
              f"(along-edge axis = {hax})")

# Temporal resolution: one timestamp per file -> monthly.
cams_times = np.atleast_1d(cams.time.values)
print(f"\nTemporal resolution: {len(cams_times)} timestamp(s) per file "
      f"(monthly; this file = {str(cams_times[0])[:10]})")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)

for ax, (var, hax) in zip(axes.ravel(), cams_edge_axis.items()):
    if var not in cams:
        ax.set_visible(False)
        continue
    da = cams[var].squeeze()
    da.plot(ax=ax, x=hax, y="height", cbar_kwargs={"label": "CH4 mole fraction (mol/mol)"})
    ax.set_title(var)
    ax.set_xlabel(f"{hax} (deg)")
    ax.set_ylabel("height (m)")

fig.suptitle(f"CAMS boundary mole fractions - {str(cams_times[0])[:10]}", fontsize=13)
plt.show()

## 5. Summary table

In [ ]:
import pandas as pd

rows = []
for var, hax in edge_horizontal_axis.items():
    if var not in ds:
        continue
    h = axis_resolution(ds, hax)
    rows.append({
        "variable": var,
        "along-edge axis": hax,
        "horizontal pts": h["n_points"],
        "horizontal step (deg)": round(h["step_median"], 4),
        "height levels": height_info["n_points"],
        "height step (m)": round(height_info["step_median"], 1),
        "height range (m)": f"{height_info['min']:.0f}-{height_info['max']:.0f}",
    })

summary = pd.DataFrame(rows)
summary

In [ ]:
ds.close()